# Programming Assignment: Model-Based Feature Selection

Welcome to the Model-Based Feature Selection Programming Assignment!

Model-based feature selection methods go beyond simple statistical filters by training machine learning models to determine which features are truly useful. Instead of measuring raw statistical relationships, these methods ask: which features help this specific model perform better? This leads to selections that are more closely aligned with the learning algorithm you plan to use.

Regularisation-based methods like LASSO automatically zero out irrelevant feature coefficients, while tree-based methods expose feature importances derived from the training process itself. Wrapper methods like RFE and Sequential Feature Selection evaluate actual subsets of features by training and scoring models, identifying combinations that work well together.

**What You Will Do in This Assignment:**

* **LASSO Feature Selection** - Apply L1 regularisation to identify non-zero coefficients and extract the most relevant features
* **Alpha Tuning** - Explore how regularisation strength controls feature sparsity and the number of selected features
* **Tree-Based Importance** - Extract and compare Gini (MDI) importance from a Random Forest classifier
* **Permutation Importance** - Evaluate feature importance on held-out data to avoid high-cardinality bias
* **Recursive Feature Elimination** - Apply RFE and check stability across bootstrap resamples
* **Sequential Feature Selection** - Run forward and backward SFS using cross-validated scoring at every step
* **Ensemble Selection** - Aggregate rankings from multiple methods for a more robust final selection
* **Model-Based Pipeline** - Build a complete sklearn Pipeline with embedded feature selection
* **Forward Selection from Scratch** - Implement greedy forward selection using cross-validated accuracy

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>
* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.
* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.
* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.
* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.
* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents
- [Imports](#setup)
- [1 - LASSO-Based Feature Selection](#ex1)
    - **[Exercise 1 - lasso_feature_selection](#ex1)**
- [2 - Tuning Regularisation Strength](#ex2)
    - **[Exercise 2 - tune_lasso_alpha](#ex2)**
- [3 - Tree-Based Feature Importance](#ex3)
    - **[Exercise 3 - get_tree_gini_importance](#ex3)**
- [4 - Permutation Importance](#ex4)
    - **[Exercise 4 - get_permutation_importance](#ex4)**
- [5 - Recursive Feature Elimination](#ex5)
    - **[Exercise 5 - apply_rfe](#ex5)**
- [6 - RFE Stability](#ex6)
    - **[Exercise 6 - check_rfe_stability](#ex6)**
- [7 - Sequential Feature Selection](#ex7)
    - **[Exercise 7 - apply_sequential_selection](#ex7)**
- [8 - Ensemble Feature Selection](#ex8)
    - **[Exercise 8 - ensemble_feature_selection](#ex8)**
- [9 - Model-Based Pipeline](#ex9)
    - **[Exercise 9 - create_model_based_pipeline](#ex9)**
- [Bonus - Forward Selection from Scratch](#bonus)
    - **[Bonus - forward_selection_from_scratch](#bonus)**

<a name="setup"></a>
## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import unittests
matplotlib.rcParams['figure.dpi'] = 100

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, LassoCV, LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import (
    SelectFromModel, RFE, RFECV,
    SequentialFeatureSelector,
    f_classif, mutual_info_classif
)
from sklearn.inspection import permutation_importance
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.utils import resample

np.random.seed(42)
print('All imports successful!')


In [ ]:
# ── Classification dataset ─────────────────────────────────────────────────
# 25 features total: 8 informative, 5 redundant, 12 noise
X_cls, y_cls = make_classification(
    n_samples=800, n_features=25, n_informative=8,
    n_redundant=5, n_repeated=0, n_clusters_per_class=2,
    random_state=42
)
FEATURE_NAMES_CLS = [f'feat_{i:02d}' for i in range(25)]

X_cls_tr, X_cls_te, y_cls_tr, y_cls_te = train_test_split(
    X_cls, y_cls, test_size=0.25, random_state=42
)

# Scale for linear models
scaler = StandardScaler()
X_cls_tr_sc = scaler.fit_transform(X_cls_tr)
X_cls_te_sc  = scaler.transform(X_cls_te)

# ── Regression dataset ──────────────────────────────────────────────────────
# 20 features: 6 informative
X_reg, y_reg = make_regression(
    n_samples=600, n_features=20, n_informative=6,
    noise=20, random_state=42
)
FEATURE_NAMES_REG = [f'rfeat_{i:02d}' for i in range(20)]

X_reg_tr, X_reg_te, y_reg_tr, y_reg_te = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

scaler_reg = StandardScaler()
X_reg_tr_sc = scaler_reg.fit_transform(X_reg_tr)
X_reg_te_sc  = scaler_reg.transform(X_reg_te)

print(f'Classification   -  train: {X_cls_tr.shape}, test: {X_cls_te.shape}')
print(f'Regression       -  train: {X_reg_tr.shape}, test: {X_reg_te.shape}')


<a name="ex1"></a>
## Exercise 1  -  LASSO Feature Selection

Implement `lasso_feature_selection` that:
1. Wraps a `Lasso(alpha=alpha)` model inside `SelectFromModel(threshold='mean')`.
2. Fits the selector on `(X_train, y_train)`.
3. Returns a **tuple** `(selector, X_selected, coef_df)` where:
   - `selector` is the fitted `SelectFromModel` object.
   - `X_selected` is `X_train` transformed by the selector.
   - `coef_df` is a `pd.DataFrame` with columns `['feature', 'coefficient']`, sorted by `abs(coefficient)` descending, containing **only selected** features.


In [ ]:
def lasso_feature_selection(X_train, y_train, alpha, feature_names):
    """
    Apply LASSO-based feature selection via SelectFromModel.

    Parameters
    ----------
    X_train : np.ndarray, shape (n_samples, n_features)
    y_train : np.ndarray, shape (n_samples,)
    alpha   : float   -  regularisation strength for Lasso
    feature_names : list of str

    Returns
    -------
    selector   : fitted SelectFromModel
    X_selected : np.ndarray, transformed training matrix
    coef_df    : pd.DataFrame, columns ['feature','coefficient'],
                 sorted by abs(coefficient) descending, selected only
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###


In [ ]:
# Test 1  -  LASSO feature selection
sel1, X_sel1, coef_df1 = lasso_feature_selection(X_reg_tr_sc, y_reg_tr, 0.1, FEATURE_NAMES_REG)
assert hasattr(sel1, 'get_support'), 'selector must be a SelectFromModel'
assert X_sel1.ndim == 2, 'X_selected must be 2-D'
assert X_sel1.shape[0] == X_reg_tr_sc.shape[0], 'row count must match'
assert X_sel1.shape[1] == sel1.get_support().sum(), 'column count must match selector'
assert list(coef_df1.columns) == ['feature', 'coefficient'], 'wrong columns'
assert len(coef_df1) == X_sel1.shape[1], 'coef_df must have one row per selected feature'
# Check sorted descending by |coef|
abs_coefs = coef_df1['coefficient'].abs().values
assert all(abs_coefs[i] >= abs_coefs[i+1] for i in range(len(abs_coefs)-1)), 'must be sorted desc'
print(f'Exercise 1 passed! Selected {X_sel1.shape[1]}/{X_reg_tr_sc.shape[1]} features.')
print(coef_df1.head())


In [ ]:
unittests.exercise_1(lasso_feature_selection)

<a name="ex2"></a>
## Exercise 2  -  Tuning LASSO Alpha

Implement `tune_lasso_alpha` that:
1. Iterates over each value in `alphas`.
2. Fits a `Lasso(alpha=a)` on `(X_train, y_train)`.
3. Counts the number of non-zero coefficients.
4. Returns a `pd.DataFrame` with columns `['alpha', 'n_selected']`.


In [ ]:
def tune_lasso_alpha(X_train, y_train, alphas):
    """
    Return a DataFrame showing how the number of selected features
    changes with alpha.

    Parameters
    ----------
    X_train : np.ndarray
    y_train : np.ndarray
    alphas  : list of float

    Returns
    -------
    pd.DataFrame with columns ['alpha', 'n_selected']
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###


In [ ]:
# Test 2  -  LASSO alpha tuning
alphas_grid = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
alpha_df = tune_lasso_alpha(X_reg_tr_sc, y_reg_tr, alphas_grid)
assert list(alpha_df.columns) == ['alpha', 'n_selected'], 'wrong columns'
assert len(alpha_df) == len(alphas_grid), 'must have one row per alpha'
assert alpha_df['n_selected'].iloc[0] >= alpha_df['n_selected'].iloc[-1], \
    'smaller alpha should retain >= features than larger alpha'
print('Exercise 2 passed!')
# Plot
plt.semilogx(alpha_df['alpha'], alpha_df['n_selected'], marker='o')
plt.xlabel('Alpha'); plt.ylabel('Features retained'); plt.title('LASSO: Alpha vs Features')
plt.grid(True); plt.tight_layout(); plt.show()


In [ ]:
unittests.exercise_2(tune_lasso_alpha)

<a name="ex3"></a>
## Exercise 3  -  Tree-Based Feature Importance (Gini)

Implement `get_tree_gini_importance` that:
1. Trains a `RandomForestClassifier(n_estimators=200, random_state=42)`.
2. Returns a `pd.DataFrame` with columns `['feature', 'importance']`,
   sorted by `importance` **descending**.


In [ ]:
def get_tree_gini_importance(X_train, y_train, feature_names):
    """
    Compute Mean Decrease in Impurity (Gini) importances.

    Returns
    -------
    pd.DataFrame with columns ['feature', 'importance'], sorted desc.
    Also returns the fitted RandomForestClassifier as second element.
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###


In [ ]:
# Test 3  -  Tree Gini importance
gini_df, rf_model = get_tree_gini_importance(X_cls_tr, y_cls_tr, FEATURE_NAMES_CLS)
assert list(gini_df.columns) == ['feature', 'importance'], 'wrong columns'
assert len(gini_df) == len(FEATURE_NAMES_CLS), 'must have one row per feature'
assert abs(gini_df['importance'].sum() - 1.0) < 1e-6, 'importances must sum to 1'
assert gini_df['importance'].iloc[0] >= gini_df['importance'].iloc[-1], 'must be sorted desc'
print(f'Exercise 3 passed! Top feature: {gini_df.iloc[0]["feature"]} ({gini_df.iloc[0]["importance"]:.4f})')
# Plot top 10
top10 = gini_df.head(10)
plt.figure(figsize=(8,4))
plt.barh(top10['feature'][::-1], top10['importance'][::-1], color='steelblue')
plt.xlabel('MDI Importance'); plt.title('Top 10 Gini Importances'); plt.tight_layout(); plt.show()


In [ ]:
unittests.exercise_3(get_tree_gini_importance)

<a name="ex4"></a>
## Exercise 4  -  Permutation Importance

Implement `get_permutation_importance` that:
1. Calls `permutation_importance(model, X_val, y_val, n_repeats=30, random_state=42, n_jobs=-1)`.
2. Returns a `pd.DataFrame` with columns `['feature', 'importance_mean', 'importance_std']`,
   sorted by `importance_mean` **descending**.


In [ ]:
def get_permutation_importance(model, X_val, y_val, feature_names):
    """
    Compute permutation importance on validation data.

    Parameters
    ----------
    model        : fitted sklearn estimator
    X_val        : np.ndarray  -  validation features
    y_val        : np.ndarray  -  validation labels
    feature_names: list of str

    Returns
    -------
    pd.DataFrame with columns ['feature','importance_mean','importance_std'],
    sorted by importance_mean descending.
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###


In [ ]:
# Test 4  -  Permutation importance
perm_df = get_permutation_importance(rf_model, X_cls_te, y_cls_te, FEATURE_NAMES_CLS)
assert list(perm_df.columns) == ['feature','importance_mean','importance_std'], 'wrong columns'
assert len(perm_df) == len(FEATURE_NAMES_CLS), 'must have one row per feature'
assert perm_df['importance_mean'].iloc[0] >= perm_df['importance_mean'].iloc[-1], 'must be sorted desc'
print('Exercise 4 passed!')
# Side-by-side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
gini_df.head(10).plot.barh(x='feature', y='importance', ax=ax1, color='steelblue', legend=False)
ax1.set_title('Gini Importance'); ax1.invert_yaxis()
perm_df.head(10).plot.barh(x='feature', y='importance_mean', ax=ax2, color='darkorange', legend=False)
ax2.set_title('Permutation Importance'); ax2.invert_yaxis()
plt.tight_layout(); plt.show()


In [ ]:
unittests.exercise_4(get_permutation_importance)

<a name="ex5"></a>
## Exercise 5  -  Recursive Feature Elimination

Implement `apply_rfe` that:
1. Creates `RFE(estimator=estimator, n_features_to_select=n_features, step=1)`.
2. Fits it on `(X_train, y_train)`.
3. Returns a **tuple** `(rfe, X_selected, ranking_df)` where:
   - `rfe` is the fitted RFE object.
   - `X_selected` is `X_train` transformed through `rfe`.
   - `ranking_df` is a `pd.DataFrame` with columns `['feature', 'rank']`,
     sorted by `rank` ascending (rank 1 = selected).


In [ ]:
def apply_rfe(X_train, y_train, estimator, n_features, feature_names):
    """
    Apply Recursive Feature Elimination.

    Returns
    -------
    rfe         : fitted RFE
    X_selected  : np.ndarray
    ranking_df  : pd.DataFrame columns ['feature','rank'], sorted by rank asc
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###


In [ ]:
# Test 5  -  RFE
from sklearn.linear_model import LogisticRegression
lr_est = LogisticRegression(max_iter=2000, random_state=42)
rfe5, X_rfe, rank_df = apply_rfe(X_cls_tr_sc, y_cls_tr, lr_est, 8, FEATURE_NAMES_CLS)
assert hasattr(rfe5, 'support_'), 'must return fitted RFE'
assert X_rfe.shape == (X_cls_tr_sc.shape[0], 8), f'expected shape ({X_cls_tr_sc.shape[0]}, 8), got {X_rfe.shape}'
assert list(rank_df.columns) == ['feature', 'rank'], 'wrong columns'
assert rank_df['rank'].iloc[0] == 1, 'first row must be rank 1 (selected)'
assert rfe5.support_.sum() == 8, 'must select exactly 8 features'
print(f'Exercise 5 passed! Selected features: {list(rank_df[rank_df["rank"]==1]["feature"])}')


In [ ]:
unittests.exercise_5(apply_rfe)

<a name="ex6"></a>
## Exercise 6  -  Checking RFE Stability

Implement `check_rfe_stability` that:
1. For each of `n_splits` bootstrap resamples of `(X, y)`, runs `RFE(estimator, n_features, step=1)`.
   Use `resample(X, y, random_state=i)` for `i` in range `n_splits`.
2. Collects the set of selected feature indices from each run.
3. Computes the **mean pairwise Jaccard similarity** across all pairs of runs.
4. Returns `(mean_jaccard, selected_sets)` as a float and list of sets.


In [ ]:
def check_rfe_stability(X, y, estimator, n_features, n_splits=20):
    """
    Measure stability of RFE selection via bootstrap resampling.

    Parameters
    ----------
    X, y        : data
    estimator   : sklearn estimator (will be cloned inside)
    n_features  : int  -  target number of features
    n_splits    : int  -  number of bootstrap runs

    Returns
    -------
    mean_jaccard   : float
    selected_sets  : list of sets of int (indices of selected features)
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###


In [ ]:
# Test 6  -  RFE stability
from sklearn.linear_model import LogisticRegression
lr_est6 = LogisticRegression(max_iter=2000, random_state=42)
jac, sets = check_rfe_stability(X_cls_tr_sc, y_cls_tr, lr_est6, 8, n_splits=10)
assert 0.0 <= jac <= 1.0, 'Jaccard must be in [0, 1]'
assert len(sets) == 10, 'must have one set per split'
assert all(isinstance(s, set) for s in sets), 'each element must be a set'
print(f'Exercise 6 passed! Mean Jaccard stability: {jac:.3f}')


In [ ]:
unittests.exercise_6(check_rfe_stability)

<a name="ex7"></a>
## Exercise 7  -  Sequential Feature Selection

Implement `apply_sequential_selection` that:
1. Creates a `SequentialFeatureSelector(estimator, n_features_to_select=n_features,
   direction=direction, scoring='accuracy', cv=3)`.
2. Fits it on `(X_train, y_train)`.
3. Returns the fitted `sfs` object and the list of selected feature names.

> **Tip:** `direction` can be `'forward'` or `'backward'`.


In [ ]:
def apply_sequential_selection(X_train, y_train, estimator,
                                n_features, feature_names,
                                direction='forward'):
    """
    Apply Sequential Feature Selection (forward or backward).

    Returns
    -------
    sfs              : fitted SequentialFeatureSelector
    selected_features: list of selected feature name strings
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###


In [ ]:
# Test 7  -  Sequential Feature Selection (forward)
from sklearn.linear_model import LogisticRegression
lr_est7 = LogisticRegression(max_iter=2000, random_state=42)
sfs7, sfs_feat = apply_sequential_selection(
    X_cls_tr_sc, y_cls_tr, lr_est7, 6, FEATURE_NAMES_CLS, direction='forward'
)
assert hasattr(sfs7, 'get_support'), 'must return fitted SFS'
assert len(sfs_feat) == 6, f'expected 6 features, got {len(sfs_feat)}'
assert all(f in FEATURE_NAMES_CLS for f in sfs_feat), 'all names must be valid'
print(f'Exercise 7 passed! Selected: {sfs_feat}')


In [ ]:
unittests.exercise_7(apply_sequential_selection)

<a name="ex8"></a>
## Exercise 8  -  Ensemble Feature Selection

Implement `ensemble_feature_selection` that:
1. Runs **three** selection methods and collects a **rank per feature** (1 = most important) from each:
   - `f_classif` (filter)
   - `RandomForestClassifier(n_estimators=100, random_state=42)` (Gini importance)
   - `LogisticRegression(penalty='l1', C=0.5, solver='liblinear', max_iter=1000)` (L1 coef)
2. Averages the three ranks.
3. Returns a `pd.DataFrame` with columns `['feature','rank_f','rank_rf','rank_l1','mean_rank']`,
   sorted by `mean_rank` ascending, and a list of the **top-k** feature names.


In [ ]:
def ensemble_feature_selection(X_train, y_train, feature_names, k=10):
    """
    Aggregate feature rankings from f_classif, RF Gini, and L1 logistic.

    Returns
    -------
    rank_df  : pd.DataFrame columns ['feature','rank_f','rank_rf','rank_l1','mean_rank']
               sorted by mean_rank ascending
    top_k    : list of str  -  the k feature names with lowest mean_rank
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###


In [ ]:
# Test 8  -  Ensemble feature selection
ens_df, top8 = ensemble_feature_selection(X_cls_tr_sc, y_cls_tr, FEATURE_NAMES_CLS, k=8)
assert list(ens_df.columns) == ['feature','rank_f','rank_rf','rank_l1','mean_rank'], 'wrong columns'
assert len(ens_df) == len(FEATURE_NAMES_CLS), 'must have one row per feature'
assert len(top8) == 8, 'top_k must have k elements'
assert ens_df['mean_rank'].iloc[0] <= ens_df['mean_rank'].iloc[-1], 'must be sorted asc'
print(f'Exercise 8 passed! Top-8 by ensemble rank: {top8}')
print(ens_df.head(8))


In [ ]:
unittests.exercise_8(ensemble_feature_selection)

<a name="ex9"></a>
## Exercise 9  -  Building a Model-Based Pipeline

Implement `create_model_based_pipeline` that:
1. Accepts `method` ∈ `{'lasso', 'rf'}` and `k` (number of features to select).
2. Builds and returns an sklearn `Pipeline` with these steps:
   - `'scaler'`: `StandardScaler()`
   - `'selector'`:
     - If `method == 'lasso'`: `SelectFromModel(Lasso(alpha=0.01), max_features=k, threshold=-np.inf)`
     - If `method == 'rf'`:   `SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42), max_features=k, threshold=-np.inf)`
   - `'model'`: `LogisticRegression(max_iter=1000)`
3. The function must raise `ValueError` for unknown `method` values.


In [ ]:
def create_model_based_pipeline(method, k):
    """
    Build a Pipeline: StandardScaler -> SelectFromModel -> LogisticRegression.

    Parameters
    ----------
    method : str   -  'lasso' or 'rf'
    k      : int   -  number of top features to keep (max_features)

    Returns
    -------
    sklearn Pipeline

    Raises
    ------
    ValueError for unknown method
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###


In [ ]:
# Test 9  -  Model-based pipeline
import sklearn

for method in ['lasso', 'rf']:
    pipe = create_model_based_pipeline(method, k=8)
    assert hasattr(pipe, 'fit'), f'{method} pipeline must have fit()'
    assert list(pipe.named_steps.keys()) == ['scaler','selector','model'], \
        'pipeline must have exactly: scaler, selector, model'
    scores = cross_val_score(pipe, X_cls, y_cls, cv=5, scoring='accuracy')
    print(f'  method={method} -> 5-fold accuracy: {scores.mean():.3f} ± {scores.std():.3f}')

try:
    create_model_based_pipeline('unknown', 5)
    assert False, 'Should have raised ValueError'
except ValueError:
    pass
print('Exercise 9 passed!')


In [ ]:
unittests.exercise_9(create_model_based_pipeline)

<a name="bonus"></a>
## Bonus Exercise  -  Forward Selection from Scratch

Implement `forward_selection_from_scratch` that:
1. Starts with an empty set of selected feature indices.
2. At each step, tries adding each **remaining** feature to the current set.
3. Evaluates the cross-validation score (accuracy) with `LogisticRegression(max_iter=1000)` and `cv=3`.
4. Permanently adds the feature that gave the **highest** CV score in that step.
5. Stops when `k` features have been selected.
6. Returns `(selected_indices, score_history)` where:
   - `selected_indices` is a list of ints (column indices in order of selection).
   - `score_history` is a list of floats (best CV score at each step).


In [ ]:
def forward_selection_from_scratch(X, y, k):
    """
    Greedy forward feature selection using cross-validated accuracy.

    Parameters
    ----------
    X : np.ndarray, shape (n_samples, n_features)
    y : np.ndarray, shape (n_samples,)
    k : int  -  number of features to select

    Returns
    -------
    selected_indices : list of int    -  indices of selected features
    score_history    : list of float  -  best CV score at each step
    """
    ### START CODE HERE ###
    pass
    ### END CODE HERE ###


In [ ]:
# Bonus test  -  Forward selection from scratch
sel_idx, scores_hist = forward_selection_from_scratch(X_cls_tr_sc, y_cls_tr, k=8)
assert len(sel_idx) == 8, 'must select exactly k features'
assert len(scores_hist) == 8, 'score_history must have k entries'
assert all(0 <= s <= 1 for s in scores_hist), 'scores must be in [0,1]'
assert len(set(sel_idx)) == 8, 'all selected indices must be unique'
print(f'Bonus passed! Selected indices: {sel_idx}')
plt.plot(range(1, 9), scores_hist, marker='o')
plt.xlabel('Number of features'); plt.ylabel('CV Accuracy')
plt.title('Forward Selection: CV Accuracy vs Feature Count')
plt.grid(True); plt.tight_layout(); plt.show()


In [ ]:
unittests.exercise_10(forward_selection_from_scratch)